# Evaluacion de modelos y dataset final

Este notebook contiene la evaluacion, seleccion del mejor modelo, regeneracion del modelo final, calculo de `score_final`, semaforo final y generacion del CSV final para la demo.

In [1]:
from __future__ import annotations

from pathlib import Path
import ast
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, auc, confusion_matrix, f1_score, precision_score, recall_score, roc_curve

pd.set_option("display.max_columns", 220)

In [2]:
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_PROCESSED = ROOT / "data" / "processed"
ARTIFACT = ROOT / "artifact"
FINAL_DIR = ARTIFACT / "final-model"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

from scripts.features.build_features import SCORE_COLUMNS
from scripts.rules.fraud_rules import apply_critical_rules
from scripts.explainability.explain_score import add_score_final, add_final_semaforo, add_explainability_columns, safe_list


In [3]:
preds = pd.read_csv(DATA_PROCESSED / "predictions.csv")
models = {
    "logistic_regression": ARTIFACT / "logistic_regression.pkl",
    "decision_tree": ARTIFACT / "decision_tree.pkl",
    "random_forest": ARTIFACT / "random_forest.pkl",
}
with open(ARTIFACT / "model_input_metadata.json", encoding="utf-8") as f:
    model_metadata = json.load(f)

preds.head()

,model,row_index,id_siniestro,y_true,y_pred,y_prob
0,logistic_regression,25,26,0,0,0.002301
1,logistic_regression,17,18,0,0,0.001625
2,logistic_regression,93,94,0,0,0.000103
3,logistic_regression,76,77,1,1,0.999993
4,logistic_regression,1,2,0,0,0.045240


In [4]:
def plot_confusion_matrix(y_true, y_pred, title_suffix: str = "") -> None:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_title(f"Matriz de confusion {title_suffix}".strip())
    ax.set_xlabel("Prediccion")
    ax.set_ylabel("Real")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


def plot_roc_curve(y_true, y_prob, title_suffix: str = "") -> float:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title(f"Curva ROC {title_suffix}".strip())
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
    return roc_auc


def evaluate_classification(y_true, y_pred, y_prob, model_name: str = "", show_plots: bool = False) -> dict[str, float]:
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    metrics["auc_roc"] = auc(fpr, tpr)
    if show_plots:
        suffix = f"- {model_name}" if model_name else ""
        plot_roc_curve(y_true, y_prob, suffix)
        plot_confusion_matrix(y_true, y_pred, suffix)
    return metrics

In [5]:
metrics_rows = []
for model_name in preds["model"].unique():
    subset = preds[preds["model"] == model_name]
    metrics = evaluate_classification(
        subset["y_true"].to_numpy(),
        subset["y_pred"].to_numpy(),
        subset["y_prob"].to_numpy(),
        model_name=model_name,
    )
    metrics["model"] = model_name
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows).sort_values(["f1_score", "recall", "auc_roc"], ascending=False)
metrics_df.to_csv(DATA_PROCESSED / "model_metrics.csv", index=False)
metrics_df

,accuracy,precision,recall,f1_score,auc_roc,model
0,1.0,1.0,1.0,1.0,1.0,logistic_regression
1,1.0,1.0,1.0,1.0,1.0,decision_tree
2,1.0,1.0,1.0,1.0,1.0,random_forest


In [6]:
best_model_name = metrics_df.iloc[0]["model"]
best_model = joblib.load(models[best_model_name])
joblib.dump(best_model, FINAL_DIR / "model.pkl")
best_model_name, FINAL_DIR / "model.pkl"

('logistic_regression',
 WindowsPath('C:/Users/H P/Desktop/fraudIA-Novo-2S1R-limpio/fraudIA-Novo-2S1R-master/artifact/final-model/model.pkl'))

In [7]:
features = pd.read_csv(DATA_PROCESSED / "features_siniestros.csv")
model_columns = model_metadata["model_input_columns"]

X_all = features.copy()
for col in model_columns:
    if col not in X_all.columns:
        X_all[col] = np.nan
X_all = X_all[model_columns]

probabilidad_ml = best_model.predict_proba(X_all)[:, 1]
prediccion_ml = (probabilidad_ml >= 0.5).astype(int)

scored = features.copy()
scored["probabilidad_ml"] = probabilidad_ml
scored["prediccion_ml"] = prediccion_ml

scored[["id_siniestro", "score_total_reglas", "probabilidad_ml", "prediccion_ml"]].head()

,id_siniestro,score_total_reglas,probabilidad_ml,prediccion_ml
0,1,24,0.006029,0
1,2,15,0.045240,0
2,3,44,0.155695,0
3,4,13,0.002448,0
4,5,13,0.000123,0


In [8]:
rules_df = apply_critical_rules(scored)
scored = pd.concat([scored.drop(columns=[c for c in rules_df.columns if c in scored.columns], errors="ignore"), rules_df], axis=1)

scored = add_score_final(scored, rule_weight=0.70, ml_weight=0.30)
scored = add_final_semaforo(scored)

scored[["id_siniestro", "score_total_reglas", "probabilidad_ml", "score_final", "semaforo_final"]].head()


,id_siniestro,score_total_reglas,probabilidad_ml,score_final,semaforo_final
0,1,24,0.006029,16.98,Verde
1,2,15,0.045240,11.86,Rojo
2,3,44,0.155695,35.47,Rojo
3,4,13,0.002448,9.17,Verde
4,5,13,0.000123,9.10,Amarillo


In [9]:
scored = add_explainability_columns(scored)
scored[["id_siniestro", "semaforo_final", "explicabilidad"]].head()


,id_siniestro,semaforo_final,explicabilidad
0,1,Verde,Caso clasificado como Verde con score final 16...
1,2,Rojo,Caso clasificado como Rojo con score final 11....
2,3,Rojo,Caso clasificado como Rojo con score final 35....
3,4,Verde,Caso clasificado como Verde con score final 9....
4,5,Amarillo,Caso clasificado como Amarillo con score final...


In [10]:
base_cols = [
    "id_siniestro", "id_poliza", "id_asegurado", "id_proveedor", "id_vehiculo",
    "ramo", "cobertura", "fecha_ocurrencia", "fecha_reporte",
    "monto_reclamado", "monto_estimado", "monto_pagado", "suma_asegurada",
    "estado", "sucursal", "ciudad_poliza", "ciudad_asegurado", "ciudad_proveedor",
    "descripcion", "documentos_completos", "docs_faltantes", "docs_inconsistentes",
    "freq_asegurado_18m", "freq_vehiculo_18m", "freq_conductor_18m",
    "max_similitud_textual", "ids_siniestros_similares_top5",
]
score_cols = SCORE_COLUMNS + ["score_total_reglas", "semaforo_score"]
model_cols = ["probabilidad_ml", "prediccion_ml", "score_final", "semaforo_score_final", "semaforo_reglas_criticas", "semaforo_final"]
rule_cols = [col for col in scored.columns if col.startswith("RF_")]
explain_cols = ["reglas_criticas_activadas", "alertas_score_activadas", "explicabilidad"]
optional_target = ["etiqueta_fraude_simulada"] if "etiqueta_fraude_simulada" in scored.columns else []

ordered_cols = [c for c in base_cols + score_cols + model_cols + rule_cols + explain_cols + optional_target if c in scored.columns]
final_df = scored.loc[:, ordered_cols].copy()
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

final_path = DATA_PROCESSED / "siniestros_scored_final.csv"
final_df.to_csv(final_path, index=False)

final_df.shape, final_path

((100, 65),
 WindowsPath('C:/Users/H P/Desktop/fraudIA-Novo-2S1R-limpio/fraudIA-Novo-2S1R-master/data/processed/siniestros_scored_final.csv'))

In [11]:
assert "score_final" in final_df.columns
assert "semaforo_final" in final_df.columns
assert "explicabilidad" in final_df.columns
assert final_df.columns.duplicated().sum() == 0
assert final_df["ids_siniestros_similares_top5"].apply(lambda value: len(safe_list(value)) <= 5).all()

final_df.head()

,id_siniestro,id_poliza,id_asegurado,id_proveedor,id_vehiculo,ramo,cobertura,fecha_ocurrencia,fecha_reporte,monto_reclamado,monto_estimado,monto_pagado,suma_asegurada,estado,sucursal,ciudad_poliza,ciudad_asegurado,ciudad_proveedor,descripcion,documentos_completos,docs_faltantes,docs_inconsistentes,freq_asegurado_18m,freq_vehiculo_18m,freq_conductor_18m,max_similitud_textual,ids_siniestros_similares_top5,score_reclamo_vigencia,score_demora_robo,score_freq_asegurado,score_freq_vehiculo,score_freq_conductor,score_rc_only,score_proveedor,score_docs_incompletos,score_docs_inconsistentes,score_dinamica_sospechosa,score_sin_tercero,score_reporte_tardio,score_monto_suma_asegurada,score_narrativas_similares,score_total_reglas,semaforo_score,probabilidad_ml,prediccion_ml,score_final,semaforo_score_final,semaforo_reglas_criticas,semaforo_final,RF_01_perdida_total_robo,RF_02_adulteracion_doc,RF_03_lista_restrictiva,RF_04_dinamica_imposible,RF_05_borde_vigencia_48h,RF_06_demora_robo_4dias,RF_07_narrativa_clonada,RF_08_score_reglas_alto,RF_09_score_alto_y_ml_riesgo,RF_10_documental_multiple,RF_11_proveedor_recurrente_monto_atipico,RF_12_alta_frecuencia_y_borde_vigencia,reglas_criticas_activadas,alertas_score_activadas,explicabilidad,etiqueta_fraude_simulada
0,1,1,50,28.0,NaN,OTRO,Cobertura especial,2026-05-11,2026-05-27,4830.07,4173.47,0.00,21580.97,CIERRE_SIN_CONSECUENCIA,Sucursal Manta,Guayaquil,Guayaquil,Cuenca,Siniestro simulado de ramo otro por cobertura ...,True,False,True,1,0,1,0.618118,"[43, 73, 14, 37, 4]",4,0,0,0,0,0,5,0,10,0,0,5,0,0,24,Verde,0.006029,0,16.98,Verde,Verde,Verde,False,False,False,False,False,False,False,False,False,False,False,False,[],"[""Siniestro cercano al inicio o fin de vigenci...",Caso clasificado como Verde con score final 16...,0
1,2,2,8,11.0,NaN,HOGAR,Responsabilidad familiar,2025-09-17,2025-09-27,2482.29,1985.12,1540.39,31712.15,PAGO_TOTAL,Matriz Guayaquil,Ambato,Daule,Ambato,Siniestro simulado de ramo hogar por cobertura...,True,False,False,0,0,0,0.407061,"[82, 3, 71, 17, 11]",0,0,0,0,0,0,10,0,0,0,0,5,0,0,15,Verde,0.045240,0,11.86,Verde,Rojo,Rojo,False,False,True,False,False,False,False,False,False,False,False,False,"[""RF_03_lista_restrictiva""]","[""Beneficiario o proveedor recurrente u observ...",Caso clasificado como Rojo con score final 11....,0
2,3,3,56,15.0,NaN,HOGAR,Robo domiciliario,2023-11-23,2023-12-09,39825.18,39104.68,38351.48,82847.76,PAGO_TOTAL,Sucursal Manta,Ambato,Ambato,Guayaquil,Siniestro simulado de ramo hogar por cobertura...,False,True,True,1,0,1,0.852134,"[33, 60, 70, 54, 82]",0,8,0,0,0,0,5,4,10,0,0,5,4,8,44,Amarillo,0.155695,0,35.47,Verde,Rojo,Rojo,False,False,False,False,False,True,True,False,False,True,True,False,"[""RF_06_demora_robo_4dias"", ""RF_07_narrativa_c...","[""Demora relevante entre ocurrencia y denuncia...",Caso clasificado como Rojo con score final 35....,0
3,4,4,13,4.0,NaN,SALUD,Hospitalización,2024-01-20,2024-01-22,27810.18,24813.58,0.00,55004.66,NEGATIVA,Sucursal Cuenca,Guayaquil,Cuenca,Santo Domingo,Siniestro simulado de ramo salud por cobertura...,True,False,False,0,0,0,0.819391,"[74, 45, 97, 64, 56]",0,0,0,0,0,0,5,0,0,0,0,0,4,4,13,Verde,0.002448,0,9.17,Verde,Verde,Verde,False,False,False,False,False,False,False,False,False,False,False,False,[],"[""Beneficiario o proveedor recurrente u observ...",Caso clasificado como Verde con score final 9....,0
4,5,5,56,26.0,NaN,VIDA,Enfermedad grave,2023-09-24,2023-10-05,4730.35,4924.49,381.04,87744.63,ANTICIPO,Sucursal Cuenca,Ambato,Ambato,Loja,Siniestro simulado de ramo vida por cobertura ...,True,False,False,0,0,0,0.859045,"[13, 88, 58, 35, 66]",0,0,0,0,0,0,0,0,0,0,0,5,0,8,13,Verde,0.000123,0,9.10,Verde,Amarillo,Amarillo,False,False,False,False,False,False,True,False,False,False,False,False,"[""RF_07_narrativa_clonada""]","[""Reporte tardio del siniestro frente a la fec...",Caso clasificado como Amarillo con score final...,0
